# Lab 16 — Cross-Validation and Learning Curves

In this lab you will test how much you can trust a single train/test split by comparing it against cross-validation's fold-averaged estimate, then diagnose whether a model suffers from high bias or high variance by reading its learning curve — using the same Titanic dataset from earlier in the course.

**Concepts covered:** k-fold cross-validation, `StratifiedKFold` for imbalanced targets, how the choice of k affects the stability of a performance estimate, learning curves, and diagnosing high bias vs. high variance from the shape of the train/validation curves.

**Reference working sessions:**
- `working-sessions/model_evaluation/04_cross_validation.ipynb`
- `working-sessions/model_evaluation/07_learning_curves.ipynb`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, learning_curve,
)

from tkh_utils import (
    PALETTE, FONT, base_layout,
    check_answer, make_answer_key, make_grading_summary,
    load_titanic,
)

_ak = make_answer_key({
    'q1': 'B',
    'q2': 'A',
    'q3': 'A',
    'q4': 'B',
})

---
## Section A — Multiple Choice

Fill in each answer variable with the letter of the best answer (A, B, C, or D).

In [ ]:
# Q1 — Why does k-fold cross-validation typically give a more trustworthy
# performance estimate than a single train/test split?
#
#   A) It trains a larger, more powerful model than a single split would
#   B) A single split gives you one score that depends on which rows
#      happened to land in the test set; cross-validation averages the
#      score across several different splits, so the estimate isn't at
#      the mercy of one lucky or unlucky split
#   C) Cross-validation always produces a higher score than a single split
#   D) Cross-validation removes the need to ever use a held-out test set
#      for anything

q1_answer = "___"  # Replace with A, B, C, or D

assert q1_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q1_answer, _ak['q1']), \
    "Not quite — revisit working-sessions/model_evaluation/04_cross_validation.ipynb " \
    "and the section comparing a single split's score to the cross-validated distribution of scores."
print("✓ Question 1 correct!")

In [ ]:
# Q2 — The Titanic dataset's survival target is imbalanced (about 38%
# survived, 62% did not). Why does that specifically call for
# StratifiedKFold instead of plain KFold?
#
#   A) Plain KFold doesn't guarantee each fold keeps roughly the same
#      38/62 survival split as the full dataset — an unlucky fold could
#      end up with far too few (or too many) survivors to evaluate the
#      model fairly. StratifiedKFold enforces that balance in every fold
#   B) Plain KFold cannot be used at all on classification problems
#   C) StratifiedKFold trains a completely different type of model
#      underneath
#   D) Class imbalance doesn't actually matter for cross-validation, only
#      for the final held-out test set

q2_answer = "___"  # Replace with A, B, C, or D

assert q2_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q2_answer, _ak['q2']), \
    "Not quite — revisit working-sessions/model_evaluation/04_cross_validation.ipynb " \
    "and the reference table comparing K-Fold, Stratified K-Fold, and Time Series Split."
print("✓ Question 2 correct!")

In [ ]:
# Q3 — A learning curve shows both the training score and the validation
# score staying low and roughly equal to each other, even as the training
# set size grows. What does this pattern indicate, and would collecting
# more data fix it?
#
#   A) High bias — the model is too simple to capture the pattern in the
#      data, so more data won't help much; a more flexible model or
#      better features would
#   B) High variance — the model is memorizing the training data, so more
#      data would fix it
#   C) This pattern means the model is perfectly tuned and ready to deploy
#   D) This pattern can only happen with tree-based models

q3_answer = "___"  # Replace with A, B, C, or D

assert q3_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q3_answer, _ak['q3']), \
    "Not quite — revisit working-sessions/model_evaluation/07_learning_curves.ipynb " \
    "and the \"Reading a learning curve\" reference table."
print("✓ Question 3 correct!")

In [ ]:
# Q4 — A learning curve shows a near-perfect training score with a much
# lower validation score, and the gap barely closes even at the largest
# training size tested. What does this pattern indicate?
#
#   A) High bias — a simpler model is needed
#   B) High variance — the model fits the training data far better than
#      it generalizes; more data, regularization, or a simpler model
#      would help close the gap
#   C) The model has too few features available to it
#   D) This is exactly what a well-fitted model's learning curve should
#      look like

q4_answer = "___"  # Replace with A, B, C, or D

assert q4_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q4_answer, _ak['q4']), \
    "Not quite — revisit working-sessions/model_evaluation/07_learning_curves.ipynb " \
    "and compare the Linear Regression vs. Decision Tree learning curve patterns."
print("✓ Question 4 correct!")

In [ ]:
make_grading_summary([
    (q1_answer, _ak['q1'], "Q1: Why cross-validation is more trustworthy than one split"),
    (q2_answer, _ak['q2'], "Q2: Why an imbalanced target calls for StratifiedKFold"),
    (q3_answer, _ak['q3'], "Q3: Diagnosing a high-bias learning curve"),
    (q4_answer, _ak['q4'], "Q4: Diagnosing a high-variance learning curve"),
], total=4)

---
## Section B — Coding Exercises

The three exercises below compare a single split to a cross-validated estimate, see how the choice of k changes that estimate, and compute a learning curve to spot overfitting directly. Run them in order, each step builds on the previous one.

Run the setup cell first.

In [ ]:
# Shared setup — run this before the exercises
# Preprocessing and train/test split for the Titanic dataset
df = load_titanic()

titanic_features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
df = df[titanic_features + ["survived"]].copy()

df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["sex"] = df["sex"].map({"male": 0, "female": 1})
df = pd.get_dummies(df, columns=["embarked"], drop_first=True)

X = df.drop(columns="survived")
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Dataset shape:", X.shape)
print("Training samples:", X_train.shape[0], " Test samples:", X_test.shape[0])
print("Survival rate:", round(y.mean(), 3))

### B1 — Single split vs. cross-validation

Compute accuracy from one train/test split, then compute a 5-fold cross-validated accuracy on the same model and full dataset, and compare the mean and spread.

In [ ]:
# B1 — Compare one train/test split's accuracy to a cross-validated estimate
log_reg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
single_split_acc = log_reg.score(___, ___)   # YOUR CODE — the test features and test target

cv_strategy = ___(n_splits=5, shuffle=True, random_state=42)   # YOUR CODE — the cross-validation splitter that preserves each fold's class balance, given this target is imbalanced
cv_scores = cross_val_score(___, X, y, cv=cv_strategy, scoring="accuracy")   # YOUR CODE — a freshly instantiated logistic regression model to fit repeatedly across folds

print(f"Single train/test split accuracy: {single_split_acc:.3f}")
print(f"5-fold CV accuracy: mean={cv_scores.mean():.3f}  std={cv_scores.std():.3f}")

# --- checks ---
assert len(cv_scores) == 5, \
    "cross_val_score with 5-fold CV should return 5 scores, one per fold"
assert cv_scores.std() < 0.15, \
    "The spread across folds should be fairly small on this dataset"
print("✓ B1 complete!")

### B2 — Choosing k

Compute the mean cross-validated accuracy for k=3, k=5, and k=10, and compare how much the estimate shifts as k changes.

In [ ]:
# B2 — See how the cross-validated estimate changes as k changes
k_values = [3, 5, 10]
mean_scores = []

for k in k_values:
    cv_k = StratifiedKFold(n_splits=___, shuffle=True, random_state=42)   # YOUR CODE — this iteration's number of folds (k)
    scores_k = cross_val_score(
        LogisticRegression(max_iter=1000), ___, ___, cv=cv_k, scoring="accuracy"
        # YOUR CODE — the full feature set and full target (cross_val_score handles the splitting itself)
    )
    mean_scores.append(scores_k.mean())

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(x=[str(k) for k in k_values], y=mean_scores, color=PALETTE["primary"], ax=ax)
ax.set_xlabel("k (number of folds)")
ax.set_ylabel("Mean CV accuracy")
ax.set_ylim(0, 1)
ax.set_title("Mean Cross-Validated Accuracy by k")
plt.tight_layout()
plt.show()

# --- checks ---
assert len(mean_scores) == 3, \
    "Should have computed a mean score for each of the 3 values of k"
assert all(0.6 < s < 1.0 for s in mean_scores), \
    "Mean CV accuracy at each k should fall in a plausible range for this dataset"
print("✓ B2 complete!")

### B3 — Learning curve for one model

Compute a learning curve for an unrestricted-depth decision tree and plot how its training and validation scores move as the training set grows.

In [ ]:
# B3 — Compute and plot a learning curve for an unrestricted-depth decision tree
train_sizes, train_scores, val_scores = learning_curve(
    ___(max_depth=None, random_state=42),   # YOUR CODE — the unrestricted-depth decision tree classifier for this exercise
    X, y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="accuracy",
    train_sizes=np.linspace(0.1, 1.0, 8),
)

train_mean = train_scores.mean(axis=1)
val_mean = ___.mean(axis=___)   # YOUR CODE — the validation scores array from above; average across the fold axis, not the training-size axis

fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(x=train_sizes, y=train_mean, label="Training score", ax=ax)
sns.lineplot(x=train_sizes, y=val_mean, label="Validation score", ax=ax)
ax.set_xlabel("Training set size")
ax.set_ylabel("Accuracy")
ax.set_title("Learning Curve — Unrestricted Decision Tree")
ax.legend()
plt.tight_layout()
plt.show()

final_gap = train_mean[-1] - val_mean[-1]
print(f"Final training score: {train_mean[-1]:.3f}   Final validation score: {val_mean[-1]:.3f}   Gap: {final_gap:.3f}")

# --- checks ---
assert train_mean[-1] > 0.9, \
    "An unrestricted-depth tree should reach a very high training score"
assert final_gap > 0.1, \
    "This model should show a sizeable train/validation gap — a sign of high variance"
print("✓ B3 complete!")

---
## Section C — Applied Problem

Compare three models — logistic regression, an unrestricted decision tree, and a random forest — by computing a learning curve for each, and see which one strikes the best balance between fitting the data and generalizing to new passengers. Fill in the blanks to run the full pipeline end to end.

In [ ]:
# Section C — Diagnosing bias vs. variance across three models

# --- Step 1: Define the candidate models ---
models_c = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree (unrestricted)": ___(max_depth=None, random_state=42),   # YOUR CODE — the unrestricted-depth decision tree classifier
    "Random Forest": ___(n_estimators=200, max_depth=6, random_state=42),   # YOUR CODE — the random forest classifier
}

cv_c = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
summary_rows = []

# --- Step 2: Run a learning curve for each model ---
for name, model in models_c.items():
    train_sizes_c, train_scores_c, val_scores_c = learning_curve(
        ___, X, y, cv=cv_c, scoring="accuracy", train_sizes=np.linspace(0.1, 1.0, 8)
        # YOUR CODE — this iteration's model from the dictionary above
    )
    train_mean_c = train_scores_c.mean(axis=1)
    val_mean_c = val_scores_c.mean(___)   # YOUR CODE — the axis representing cv folds, not training set sizes
    summary_rows.append({
        "model": name,
        "final_train_score": train_mean_c[-1],
        "final_val_score": val_mean_c[-1],
        "gap": train_mean_c[-1] - val_mean_c[-1],
    })

# --- Step 3: Compare all three ---
summary_c = pd.DataFrame(summary_rows)
print(summary_c.round(3))

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=summary_c, x="model", y="gap", color=PALETTE["secondary"], ax=ax)
ax.set_ylabel("Train − validation gap")
ax.set_title("Bias/Variance Gap by Model")
plt.tight_layout()
plt.show()

forest_row = summary_c[summary_c["model"] == "Random Forest"].iloc[0]
tree_row = summary_c[summary_c["model"] == "Decision Tree (unrestricted)"].iloc[0]
logreg_row = summary_c[summary_c["model"] == "Logistic Regression"].iloc[0]

print(f"Logistic Regression — final val score: {logreg_row['final_val_score']:.3f}, gap: {logreg_row['gap']:.3f}")
print(f"Decision Tree (unrestricted) — final val score: {tree_row['final_val_score']:.3f}, gap: {tree_row['gap']:.3f}")
print(f"Random Forest — final val score: {forest_row['final_val_score']:.3f}, gap: {forest_row['gap']:.3f}")

# --- checks ---
assert forest_row["gap"] < tree_row["gap"], \
    "The random forest's train/validation gap should be smaller than the unrestricted tree's"
assert forest_row["final_val_score"] >= logreg_row["final_val_score"] - 0.02, \
    "The random forest's validation score should be roughly as good as, or better than, logistic regression's"
print("✓ Section C complete!")

---

## Section D — Reflection

These questions are for reflection. Edit the markdown cells below each question to write your response. There are no wrong answers, we are looking for thoughtful engagement with what you have learned. Your instructor may review these.

**Question D1**

Imagine you only had a small Titanic-like dataset — around 200 rows instead of 891. Would you trust a single train/test split or cross-validation more in that situation, and why?

*Your response here...*

**Question D2**

Look back at Section C's three gap values. Which model's pattern looks closest to the high-bias pattern from Q3, and which looks closest to the high-variance pattern from Q4? If the high-bias-looking model's validation score is genuinely too low for deployment, would collecting 10x more Titanic-like data fix it? What would you try instead?

*Your response here...*

**Question D3**

Describe a project (real or hypothetical) where you'd want nested cross-validation — an outer loop to evaluate the model and an inner loop to tune its hyperparameters — rather than a single round of plain k-fold cross-validation. Why would the two loops need to stay separate?

*Your response here...*